# NOTE
- Drag and drop the extracted netlist (`extracted_netlist__2026_08_10.spice`) from PC
- Gemini convo for building the parser: See https://share.gemini.google/eWX3TkKSkHWL
Building the parser

# Basic parser for top level
We just want to examine the structure of the top-level block

In [ ]:
def parse_spice(spice_text):
    # Step 1: Join continuation lines ('+') and strip empty/comment lines
    raw_lines = spice_text.splitlines()
    joined_lines = []
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith("*"):
            continue
        if line.startswith("+"):
            if joined_lines:
                joined_lines[-1] += " " + line[1:].strip()
        else:
            joined_lines.append(line)

    # Step 2: Build the hierarchy tree
    tree = {"subckts": {}, "top_instances": []}
    current_subckt = None

    for line in joined_lines:
        tokens = line.split()
        cmd = tokens[0].lower()

        # Begin subcircuit definition
        if cmd == ".subckt":
            subckt_name = tokens[1]
            pins = tokens[2:]
            current_subckt = subckt_name
            tree["subckts"][subckt_name] = {"pins": pins, "instances": []}

        # End subcircuit definition
        elif cmd == ".ends":
            current_subckt = None

        # Component/Cell Instantiation (lines starting with X or M)
        elif tokens[0].upper().startswith(("X", "M")):
            inst_name = tokens[0]

            nets_and_cell = []
            params = {}

            # Separate pin connections/cell type from 'key=value' parameters
            for token in tokens[1:]:
                if "=" in token:
                    k, v = token.split("=", 1)
                    params[k] = v
                else:
                    nets_and_cell.append(token)

            # Last non-parameter token is the subckt/primitive type
            cell_type = nets_and_cell[-1] if nets_and_cell else None
            connections = nets_and_cell[:-1]

            inst_dict = {
                "name": inst_name,
                "type": cell_type,
                "connections": connections,
                "params": params,
            }

            if current_subckt:
                tree["subckts"][current_subckt]["instances"].append(inst_dict)
            else:
                tree["top_instances"].append(inst_dict)

    return tree

In [ ]:
'''
We just want the top level netlist of the 'puzzle' block
.subckt sky130_fd_sc_hd__or4bb_2 A X B D_N C_N VPWR VGND VPB VNB
'''
with open(r'/content/extracted_netlist__2026_08_10.spice') as f:
    text = f.read()
    idx=text.find('.subckt puzzle')
    top_netlist = text[idx:]
# top_netlist
# Just a dict for quick cell lookup based on name
tree = parse_spice(top_netlist)
# print(tree)
cell_lookup = {cell_dict['name']:cell_dict for cell_dict in tree['subckts']['puzzle']['instances']}


Pick out just one instance

In [ ]:
tree['subckts']['puzzle']['instances'][0]

In [ ]:
cell_lookup = {cell_dict['name']:cell_dict for cell_dict in tree['subckts']['puzzle']['instances']}
#cell_lookup['Xsky130_fd_sc_hd__a211oi_2_1']

# .index('Xsky130_fd_sc_hd__nor4_2_0')

In [ ]:
cell_lookup[list(cell_lookup.keys())[0]]

In [ ]:
#cell_lookup['Xsky130_fd_sc_hd__nor4_2_0']

In [ ]:
len(cell_lookup)

- Flat netlist: `puzzle` is flat (has no additional level of hierarchy beyind just the standard cells) with 942 cells (no subckts)
- There are 68 standard cell subckts used in building the `puzzle` top cell
- Leading X in cellname declarations: Declarations start with 'X' and all other mentions of the same cell in the netlist dont hav this leading 'X'. eg.
The declaration for this 9 fanout MUX is:
```
Xsky130_fd_sc_hd__mux2_1_9 sky130_fd_sc_hd__inv_2_7/A sky130_fd_sc_hd__mux2_1_9/A1
```
And all other mentions of this particular MUX go without the X
```
Xsky130_fd_sc_hd__mux2_1_16 sky130_fd_sc_hd__inv_2_7/A sky130_fd_sc_hd__mux2_1_9/A0 <---- here is our MUX!
```
(Maybe this leading X is a ngspice convention?...)

# Plan of attack
- The standard cells used have been dumped to a Google sheet.
- Provide the following to a LLM:
  - The (above) list of standard cells
  - URL to lookup what these cells do (eg. https://mithro-skywater.readthedocs.io/en/latest/contents/libraries/sky130_fd_sc_hd/cells/nor4/README.html)
  - LLM Output should be the verilog for these cells




In [ ]:
# Dump to Google sheet https://docs.google.com/spreadsheets/d/1AoB2Vl69x24Fa7iI1HcYSEEqZwW_QbxtuwxnzDZMvLQ/edit?gid=0#gid=0
!grep 'subckt' extracted_netlist__2026_08_10.spice | column -t > subskts.csv

# Spice --> Verilog: Converting the instantiation lines to verilog

(First just convert and see, if we need to simulate it we need to add in the headers for standard cells from the sky130 repo)

In [ ]:
# Quick snippet to emit Verilog from your parsed dict
verilog_lines = ["module top_level ( ... );\n"]

# for inst in tree["top_instances"]:
for inst in tree['subckts']['puzzle']['instances']: # WIP
    # print(inst)
    cell_type = inst["type"]
    inst_name = inst["name"]
    pins = ", ".join(inst["connections"])

    # Emits: sky130_fd_sc_hd__nor2_2 Xinst1 (net1, net2, net3, ...);
    verilog_lines.append(f"  {cell_type} {inst_name} ({pins});\n")

verilog_lines.append("endmodule")

In [ ]:
subckt_dict = {}
with open('/content/subskts.csv', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2 and parts[0].lower() == '.subckt':
            subckt_name = parts[1]
            # Parameters are all tokens after the subckt name
            parameters = parts[2:]
            subckt_dict[subckt_name] = parameters

# You can print the dictionary to inspect it
# print(subckt_dict)
# print(f"Number of subcircuits found: {len(subckt_dict)}")

In [ ]:
subckt_dict

In [ ]:
verilog_lines[:10]

In [ ]:
f = open('top_module.v','w')
for v in verilog_lines:
  f.write(v)


In [ ]:
verilog_lines = []
for k,v in subckt_dict.items():
  verilog_lines.append((f'`include "{k}.v"\n'))
verilog_lines += ["module puzzle (\ninput I,\noutput O[7:0],\ninput clk,\ninput enable,\n input rst_n,\noutput success\n);"]

for inst in tree['subckts']['puzzle']['instances']: # WIP
    cell_type = inst["type"]
    inst_name = inst["name"]
    pins = ", ".join(inst["connections"])
    pins_dict = {k: v for k, v in zip(subckt_dict[cell_type], pins.split())}

    pins_dict = {k: v.replace('/','_') for k,v in pins_dict.items()}
    pins_dict = {k: v.replace(',','') for k,v in pins_dict.items()}
    pins_dict = {k: v for k,v in pins_dict.items() if k[0] != 'V'}
    if len(list(pins_dict.values())) > 0:
      pins = ", ".join([f".{k}({v})\n" for k,v in pins_dict.items()])
      verilog_lines.append(f"  {cell_type} {inst_name} ({pins});\n")

verilog_lines.append("endmodule")

In [ ]:
verilog_lines[:100]

In [ ]:
f = open('puzzle.v','w')
for v in verilog_lines:
  f.write(v)


In [ ]:
verilog_lines[-10:]

In [ ]:
!tail -f top_module.v

  sky130_fd_sc_hd__xnor2_2 Xsky130_fd_sc_hd__xnor2_2_23 (sky130_fd_sc_hd__or2_2_12/B, sky130_fd_sc_hd__a22o_2_4/B2, sky130_fd_sc_hd__xnor2_2_26/A, VGND, VPWR, VPWR, VGND);
  sky130_fd_sc_hd__and4bb_2 Xsky130_fd_sc_hd__and4bb_2_6 (sky130_fd_sc_hd__or4_2_4/A, sky130_fd_sc_hd__or4_2_4/D, sky130_fd_sc_hd__or4_2_4/C, sky130_fd_sc_hd__inv_2_9/A, sky130_fd_sc_hd__or4_2_4/B, VGND, VPWR, VPWR, VGND);
  sky130_fd_sc_hd__xnor2_2 Xsky130_fd_sc_hd__xnor2_2_12 (sky130_fd_sc_hd__nor2_2_31/B, sky130_fd_sc_hd__or4_2_4/A, sky130_fd_sc_hd__inv_2_7/A, VGND, VPWR, VPWR, VGND);
  sky130_fd_sc_hd__clkbuf_8 Xsky130_fd_sc_hd__clkbuf_8_14 (sky130_fd_sc_hd__dfxtp_2_3/CLK, sky130_fd_sc_hd__clkbuf_8_9/A, VGND, VPWR, sky130_fd_sc_hd__clkbuf_8_14/VPB, VGND);
  sky130_fd_sc_hd__decap_3 Xsky130_fd_sc_hd__decap_3_109 (VPWR, VGND, sky130_fd_sc_hd__decap_3_111/VPB, VGND);
  sky130_fd_sc_hd__nor3b_2 Xsky130_fd_sc_hd__nor3b_2_1 (sky130_fd_sc_hd__or2_2_9/A, sky130_fd_sc_hd__o31a_2_2/A3, sky130_fd_sc_hd__or3b_2_0/A, sky130_f

In [ ]:
!tail -f top_module.v

In [ ]:
!wc -l top_module.v